# 📔 个人日记系统

依次运行下方单元格启动日记界面。

In [1]:
import os
from pathlib import Path
from datetime import datetime, date, timedelta
from IPython.display import display, HTML, clear_output
import ipywidgets as widgets

BASE = Path.cwd() / 'entries'
TEMPLATE = Path.cwd() / '_template.md'

WEEKDAY_CN = ['星期一','星期二','星期三','星期四','星期五','星期六','星期日']

def _path(d):
    return BASE / str(d.year) / f'{d.month:02d}' / f'{d.strftime("%Y-%m-%d")}.md'

def read_entry(d):
    p = _path(d)
    return p.read_text(encoding='utf-8') if p.exists() else None

def write_entry(d, text):
    p = _path(d)
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(text, encoding='utf-8')

def delete_entry(d):
    p = _path(d)
    if p.exists(): p.unlink()

def list_entry_dates():
    return sorted({f.stem for f in BASE.rglob('*.md')})

def get_stats():
    entries = list_entry_dates()
    today = date.today()
    total = len(entries)
    this_year = sum(1 for e in entries if e.startswith(str(today.year)))
    this_month = sum(1 for e in entries if e.startswith(today.strftime('%Y-%m')))
    streak = 0
    d = today
    while d.strftime('%Y-%m-%d') in entries:
        streak += 1
        d -= timedelta(days=1)
    return total, this_year, this_month, streak

def render_md(text):
    if not text:
        return '<p style="color:#999;">暂无内容</p>'
    import markdown
    return markdown.markdown(text, extensions=['extra'])

print('✅ 模块加载完成')

✅ 模块加载完成


In [2]:
CSS = '''
<style>
:root {
  --bg: #f5f3f0;
  --card: #ffffff;
  --primary: #8b5cf6;
  --primary-light: #ede9fe;
  --text: #1f2937;
  --text-secondary: #6b7280;
  --border: #e5e7eb;
  --success: #10b981;
  --danger: #ef4444;
  --radius: 16px;
  --shadow: 0 4px 24px rgba(0,0,0,0.06);
}
.widget-label { font-size: 13px !important; color: var(--text-secondary) !important; }
.main-container { max-width: 960px; margin: 0 auto; padding: 20px; }
.header { display: flex; align-items: center; justify-content: space-between; margin-bottom: 20px; }
.header h1 { font-size: 26px; font-weight: 700; margin: 0; }
.header h1 span { background: linear-gradient(135deg, #8b5cf6, #ec4899); -webkit-background-clip: text; -webkit-text-fill-color: transparent; }
.card { background: var(--card); border-radius: var(--radius); box-shadow: var(--shadow); padding: 20px; margin-bottom: 14px; border: 1px solid var(--border); }
.card-title { font-size: 13px; font-weight: 600; color: var(--text-secondary); text-transform: uppercase; letter-spacing: 0.5px; margin: 0 0 10px 0; }
.stats-grid { display: grid; grid-template-columns: repeat(4, 1fr); gap: 10px; }
.stat-item { text-align: center; padding: 14px; background: var(--bg); border-radius: 12px; }
.stat-item .num { font-size: 26px; font-weight: 700; color: var(--primary); }
.stat-item .label { font-size: 12px; color: var(--text-secondary); margin-top: 2px; }
.nav-row { display: flex; align-items: center; gap: 8px; flex-wrap: wrap; }
.date-label { font-size: 17px; font-weight: 600; color: var(--text); margin: 0 8px; }
.recent-grid { display: flex; gap: 6px; flex-wrap: wrap; }
.recent-chip { padding: 4px 12px; border-radius: 8px; font-size: 12px; background: var(--bg); border: 1px solid var(--border); }
.recent-chip.done { background: var(--primary-light); border-color: var(--primary); color: var(--primary); font-weight: 500; }
.recent-chip.empty { opacity: 0.45; }
.section-label { font-size: 14px; font-weight: 600; color: var(--text); margin: 0 0 8px 0; }
textarea { font-family: 'SF Mono', 'Fira Code', monospace !important; font-size: 14px !important; line-height: 1.7 !important; }
</style>
'''
display(HTML(CSS))

In [3]:
# ---------- 构建界面 ----------

# 日期状态
cur = date.today()

# 控件
date_picker = widgets.DatePicker(value=cur, description='📅')
date_picker.style.description_width = '30px'

editor = widgets.Textarea(
    value='',
    placeholder='今天发生了什么？写下你的想法...',
    layout=widgets.Layout(width='100%', height='320px')
)

# 输出区域
stats_out = widgets.HTML()
nav_out = widgets.HTML()
preview_out = widgets.HTML()
recent_out = widgets.HTML()
toast_out = widgets.HTML()

# ---------- 核心刷新 ----------
def refresh(d=None):
    global cur
    if d:
        cur = d
    date_picker.value = cur
    
    # 加载内容
    content = read_entry(cur)
    if content:
        editor.value = content
    else:
        if TEMPLATE.exists():
            tpl = TEMPLATE.read_text(encoding='utf-8')
            tpl = tpl.replace('{{date}}', cur.strftime('%Y-%m-%d'))
            tpl = tpl.replace('{{weekday}}', WEEKDAY_CN[cur.weekday()])
            editor.value = tpl
        else:
            editor.value = f'# {cur.strftime("%Y-%m-%d")} {WEEKDAY_CN[cur.weekday()]}\n\n'
    
    # 导航
    is_today = (cur == date.today())
    weekday = WEEKDAY_CN[cur.weekday()]
    nav_out.value = f'''
    <div class="card">
      <div class="nav-row">
        <span class="date-label">{cur.strftime('%Y-%m-%d')} {weekday}</span>
      </div>
    </div>'''
    
    # 统计
    total, ty, tm, streak = get_stats()
    stats_out.value = f'''
    <div class="card">
      <div class="stats-grid">
        <div class="stat-item"><div class="num">{total}</div><div class="label">总篇数</div></div>
        <div class="stat-item"><div class="num">{ty}</div><div class="label">今年</div></div>
        <div class="stat-item"><div class="num">{tm}</div><div class="label">本月</div></div>
        <div class="stat-item"><div class="num">{streak}</div><div class="label">连续天数</div></div>
    </div></div>'''
    
    # 预览
    preview_out.value = f'<div class="card"><div class="card-title">📖 预览</div><div style="line-height:1.8">{render_md(editor.value)}</div></div>'
    
    # 最近
    entries = list_entry_dates()
    chips = []
    today_str = date.today().strftime('%Y-%m-%d')
    for i in range(13, -1, -1):
        d = date.today() - timedelta(days=i)
        ds = d.strftime('%Y-%m-%d')
        day_cn = WEEKDAY_CN[d.weekday()]
        exists = ds in entries
        cls = 'done' if exists else 'empty'
        short = f'{d.month}/{d.day}'
        chips.append(f'<span class="recent-chip {cls}" title="{ds}">{short} {day_cn[-1]}{"✓" if exists else "○"}</span>')
    recent_out.value = f'<div class="card"><div class="card-title">📅 最近14天</div><div class="recent-grid">{" ".join(chips)}</div></div>'
    
    toast_out.value = ''

def toast(msg, color='#1f2937'):
    toast_out.value = f'<div style="position:fixed;bottom:24px;right:24px;background:{color};color:#fff;padding:12px 24px;border-radius:10px;font-size:14px;z-index:999;box-shadow:0 4px 20px rgba(0,0,0,0.15);animation:fadeIn 0.3s">{msg}</div>'

display(HTML('<style>@keyframes fadeIn{from{opacity:0;transform:translateY(20px)}to{opacity:1;transform:translateY(0)}}</style>'))

print('✅ 界面已初始化')

✅ 界面已初始化


In [4]:
# ---------- 交互逻辑 ----------

# 导航按钮
prev_btn = widgets.Button(
    description='‹ 前一天',
    layout=widgets.Layout(width='120px'),
    button_style=''
)

today_btn = widgets.Button(
    description='📅 今天',
    layout=widgets.Layout(width='100px'),
    button_style='primary'
)

next_btn = widgets.Button(
    description='后一天 ›',
    layout=widgets.Layout(width='120px'),
    button_style=''
)

def go_prev(b):
    refresh(cur - timedelta(days=1))

def go_today(b):
    refresh(date.today())

def go_next(b):
    refresh(cur + timedelta(days=1))

prev_btn.on_click(go_prev)
today_btn.on_click(go_today)
next_btn.on_click(go_next)

date_picker.observe(lambda c: refresh(c['new']) if c['new'] else None, names='value')
editor.observe(lambda c: preview_out.__setattr__('value', f'<div class="card"><div class="card-title">📖 预览</div><div style="line-height:1.8">{render_md(c["new"])}</div></div>'), names='value')

# 操作按钮
save_btn = widgets.Button(
    description='💾 保存',
    layout=widgets.Layout(width='120px'),
    button_style='success'
)

delete_btn = widgets.Button(
    description='🗑 删除',
    layout=widgets.Layout(width='120px'),
    button_style='danger'
)

def on_save(b):
    write_entry(cur, editor.value)
    refresh(cur)
    toast('✅ 已保存', '#10b981')

def on_delete(b):
    if cur == date.today():
        toast('⚠️ 不能删除今天的记录', '#ef4444')
        return
    delete_entry(cur)
    refresh(cur)
    toast('🗑 已删除', '#ef4444')

save_btn.on_click(on_save)
delete_btn.on_click(on_delete)

print('✅ 交互已绑定')

✅ 交互已绑定


In [5]:
# ---------- 组装并显示 ----------

header = widgets.HTML('''
<div class="header">
  <h1>📔 <span>个人日记</span></h1>
</div>''')

nav_row = widgets.HBox([prev_btn, today_btn, next_btn, date_picker], layout=widgets.Layout(gap='8px', align_items='center'))

actions = widgets.HBox([save_btn, delete_btn], layout=widgets.Layout(gap='8px'))

editor_box = widgets.VBox([
    widgets.HTML('<div class="section-label">✏️ 编辑</div>'),
    editor
])

main = widgets.VBox([
    header,
    stats_out,
    nav_row,
    nav_out,
    actions,
    editor_box,
    preview_out,
    recent_out,
    toast_out
], layout=widgets.Layout(margin='0 auto', max_width='960px'))

display(main)
refresh(date.today())
print('\n✨ 日记已就绪，开始记录吧！')


✨ 日记已就绪，开始记录吧！
